# 02 — OCR Extraction Pipeline

**Constituency:** ชัยภูมิ เขต 2 · OCR engine: **Typhoon OCR** (`api.opentyphoon.ai`)

## Architecture: OCR-driven page dispatch

The Drive PDFs are organized **by ตำบล** — one file contains many stations stacked together. Page counts per file are not always reliable (some ตำบล have +N extra pages from re-counts), and some files mix 5/18 and 5/18 (บช) together.

Instead of mechanically splitting first then OCR\'ing each chunk, we do the opposite:

```
For each source PDF:
  1. Render every page → image
  2. OCR each page once
  3. Use the page header (หน่วยที่ X, ตำบล Y, อำเภอ Z) to identify station + form
  4. Group pages into chunks per (station × form)
  5. Parse the chunk as a unit (header + vote table)
  6. Validate (Thai-word ↔ digit, sum checks)
  7. Save to canonical schema
```

**Why this works:** each form\'s page 1 always has a clear header naming the station. We don\'t need to trust page counts.

## Output schema

### `stations.parquet` — 1 row per (station × form)
| col | type | description |
|---|---|---|
| `province`, `constituency_no` | str/int | "ชัยภูมิ", 2 |
| `form_type` | str | `5_18`, `5_18_party`, `5_16`, etc. |
| `station_code`, `station_no` | str | for 5/18 forms only |
| `district`, `subdistrict` | str | for 5/18 forms only |
| `eligible_voters`, `voters_present` | int | header counts |
| `ballots_received`, `ballots_used` | int | header counts |
| `ballots_good`, `ballots_spoiled`, `ballots_no_vote` | int | breakdown |
| `source_pdf`, `source_pages` | str | provenance |
| `ocr_status` | str | `ok` / `needs_review` / `failed` |
| `validation_flags` | str | semicolon-joined issues |

### `votes.parquet` — long format, 1 row per vote entity
| col | type | description |
|---|---|---|
| (join keys above) | | |
| `entity_kind` | str | `candidate` or `party` |
| `entity_no`, `entity_name` | int/str | from reference roster |
| `votes` | int | digit value |
| `votes_thai_word` | str | raw OCR for audit |
| `needs_review` | bool | digit ≠ word |


## 1 · Setup

In [20]:
# !pip install pdf2image pillow opencv-python beautifulsoup4 pythainlp pandas pyarrow requests --quiet
# !apt-get install -y poppler-utils    # Linux/Colab

In [21]:
#!pip install pdf2image pillow opencv-python beautifulsoup4 pythainlp pandas pyarrow requests

In [22]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads TYPHOON_API_KEY from .env in project root

True

In [23]:
import os, json, hashlib, logging
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
import requests
from pypdf import PdfReader

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s | %(message)s")
log = logging.getLogger("nb01")

# Paths
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

GDRIVE_DIR     = PROJECT_ROOT / "data" / "raw" / "pdfs_gdrive"
EXTERNAL_DIR   = PROJECT_ROOT / "data" / "external"
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

# Constituency metadata
PROVINCE_NAME      = "ชัยภูมิ"
PROVINCE_CODE      = "36"
CONSTITUENCY_NO    = 2
EXPECTED_STATIONS  = 341

print(f"Project root: {PROJECT_ROOT}")
print(f"PDF source:   {GDRIVE_DIR}")
print(f"External:     {EXTERNAL_DIR}")

Project root: /Users/pornmongkol/Downloads/final_dsde/ProjectDSDE_election2026
PDF source:   /Users/pornmongkol/Downloads/final_dsde/ProjectDSDE_election2026/data/raw/pdfs_gdrive
External:     /Users/pornmongkol/Downloads/final_dsde/ProjectDSDE_election2026/data/external


In [24]:
from pdf2image import convert_from_path
from pathlib import Path

# Test with one of your PDFs
test_pdf = next((PROJECT_ROOT / "data" / "raw" / "pdfs_gdrive").rglob("*.PDF"))
print(f"Testing: {test_pdf.name}")
pages = convert_from_path(str(test_pdf), dpi=100, first_page=1, last_page=1)
print(f"✓ pdf2image works! Page 1 size: {pages[0].size}")

Testing: กะฮาด-001-แบ่งเขต.PDF
✓ pdf2image works! Page 1 size: (825, 1065)


In [25]:
import os, json, re, time, base64, hashlib, logging
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import requests
from PIL import Image
from bs4 import BeautifulSoup

try:
    import cv2
    HAS_CV2 = True
except ImportError:
    HAS_CV2 = False

try:
    from pdf2image import convert_from_path
except ImportError:
    raise ImportError("pdf2image required: pip install pdf2image && apt-get install poppler-utils")

try:
    from pythainlp.util import text_to_num
    HAS_PYTHAINLP = True
except ImportError:
    HAS_PYTHAINLP = False
    print("⚠️  pythainlp not installed — Thai number cross-check disabled")

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s | %(message)s")
log = logging.getLogger("ocr")


## 2 · Configuration

In [26]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

EXTERNAL_DIR   = PROJECT_ROOT / "data" / "external"
IMG_CACHE_DIR  = PROJECT_ROOT / "data" / "raw" / "images"
PROCESSED_DIR  = PROJECT_ROOT / "data" / "processed"
OCR_CACHE_DIR  = PROCESSED_DIR / "ocr_cache"     # 1 file per page (raw OCR text)
PAGE_INDEX_DIR = PROCESSED_DIR / "page_index"    # 1 file per source PDF (parsed headers)
STATION_OUT    = PROCESSED_DIR / "stations_raw"  # 1 JSON per (station × form)

for p in [IMG_CACHE_DIR, PROCESSED_DIR, OCR_CACHE_DIR, PAGE_INDEX_DIR, STATION_OUT]:
    p.mkdir(parents=True, exist_ok=True)

PROVINCE         = "ชัยภูมิ"
CONSTITUENCY_NO  = 2

# Typhoon OCR. The key is checked lazily inside _typhoon_request so Run All can
# still inspect cached/aggregated data without making OCR calls.
TYPHOON_API_KEY = os.getenv("TYPHOON_API_KEY")

TYPHOON_URL = "https://api.opentyphoon.ai/v1/ocr"
TYPHOON_PARAMS = {
    "model": "typhoon-ocr",
    "task_type": "default",
    "max_tokens": 16384,
    "temperature": 0.1,
    "top_p": 0.6,
    "repetition_penalty": 1.2,
}

print(f"Project root:  {PROJECT_ROOT}")
print(f"OCR cache:     {OCR_CACHE_DIR}")
print(f"Page index:    {PAGE_INDEX_DIR}")
print(f"Station out:   {STATION_OUT}")


Project root:  /Users/pornmongkol/Downloads/final_dsde/ProjectDSDE_election2026
OCR cache:     /Users/pornmongkol/Downloads/final_dsde/ProjectDSDE_election2026/data/processed/ocr_cache
Page index:    /Users/pornmongkol/Downloads/final_dsde/ProjectDSDE_election2026/data/processed/page_index
Station out:   /Users/pornmongkol/Downloads/final_dsde/ProjectDSDE_election2026/data/processed/stations_raw


## 3 · Load reference data from nb01

In [27]:
stations_df  = pd.read_csv(EXTERNAL_DIR / "stations.csv",  encoding="utf-8-sig")
candidates_df = pd.read_csv(EXTERNAL_DIR / "candidates.csv", encoding="utf-8-sig")
parties_df   = pd.read_csv(EXTERNAL_DIR / "parties.csv",    encoding="utf-8-sig")
manifest_df  = pd.read_csv(EXTERNAL_DIR / "source_manifest.csv", encoding="utf-8-sig")

# Build lookups
CANDIDATES = {int(r["candidate_no"]): (r["candidate_name"], r["party_name"])
              for _, r in candidates_df.iterrows()}
PARTIES = {int(r["party_no"]): r["party_name"]
           for _, r in parties_df.iterrows()}

# Tambol → list of station_codes (in order)
TAMBOL_STATIONS = {
    t: g.sort_values("station_no")["station_code"].tolist()
    for t, g in stations_df.groupby("subdistrict")
}

print(f"Loaded {len(stations_df)} stations, {len(CANDIDATES)} candidates, {len(PARTIES)} parties")
print(f"Manifest: {len(manifest_df)} (station × form) entries")
print()
print("Source kind breakdown:")
print(manifest_df["source_kind"].value_counts().to_string())


Loaded 341 stations, 7 candidates, 57 parties
Manifest: 686 (station × form) entries

Source kind breakdown:
source_kind
mixed                 428
pure                  204
missing                28
multi_tambol_mixed     22
constituency            4


## 4 · PDF → Image preprocessing

Each PDF page → cached JPG. We crop the top header area and (if cv2 available) lightly denoise. Cache key includes DPI + crop params so re-running with different settings doesn\'t collide.


In [28]:
def render_page(pdf_path: Path, page_idx: int, dpi: int = 200) -> Path:
    """Render one PDF page → cached JPG. Idempotent.

    Uses ASCII-only filename for the cached image (hash of source path)
    to avoid Windows/OneDrive issues with Thai characters in filenames.
    """
    # Hash includes the full source path so different PDFs don\'t collide
    key = hashlib.md5(f"{pdf_path}|{page_idx}|{dpi}".encode("utf-8")).hexdigest()[:16]
    out = IMG_CACHE_DIR / f"page_{key}.jpg"
    if out.exists():
        return out

    pages = convert_from_path(str(pdf_path), dpi=dpi,
                              first_page=page_idx + 1, last_page=page_idx + 1)
    if not pages:
        raise IndexError(f"{pdf_path.name} has no page {page_idx}")
    arr = np.array(pages[0])

    if HAS_CV2:
        gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
        # Mild denoise — preserve thin Thai strokes
        denoised = cv2.fastNlMeansDenoising(gray, h=8)
        bgr = cv2.cvtColor(denoised, cv2.COLOR_GRAY2BGR)
        # cv2.imwrite doesn\'t handle non-ASCII paths on Windows reliably — use imencode + write
        success, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, 92])
        if not success:
            raise RuntimeError(f"cv2.imencode failed for {out}")
        with open(out, "wb") as f:
            f.write(buf.tobytes())
    else:
        Image.fromarray(arr).save(out, "JPEG", quality=92)

    if not out.exists():
        raise FileNotFoundError(f"Render claimed success but {out} doesn\'t exist on disk")

    return out


def count_pdf_pages(pdf_path: Path) -> int:
    """Cheap page count without rendering."""
    from pypdf import PdfReader
    return len(PdfReader(str(pdf_path)).pages)


## 5 · Typhoon OCR client (cached, retry-safe)

In [29]:
class TyphoonError(Exception):
    pass


def _extract_text_from_response(payload: dict) -> str:
    """
    Pull the OCR text out of Typhoon's response. Their actual shape is:
        { "results": [ { "success": True, "message": { "choices": [ {"message": {"content": "..."}} ] } } ] }

    We look at all the obvious places, in order of preference.
    """
    # Real Typhoon shape (results[0].message.choices[0].message.content)
    results = payload.get("results")
    if isinstance(results, list) and results:
        first = results[0]
        if isinstance(first, dict):
            if first.get("success") is False:
                raise TyphoonError(f"Typhoon reported failure: {first.get('error', first)}")
            msg = first.get("message", {})
            if isinstance(msg, dict):
                choices = msg.get("choices")
                if isinstance(choices, list) and choices:
                    content = choices[0].get("message", {}).get("content")
                    if content:
                        return content

    # Other shapes seen in alternative endpoints
    if "text" in payload and isinstance(payload["text"], str):
        return payload["text"]
    if "choices" in payload and isinstance(payload["choices"], list) and payload["choices"]:
        content = payload["choices"][0].get("message", {}).get("content")
        if content:
            return content

    raise TyphoonError(f"Could not extract text from payload: keys={list(payload.keys())[:10]}")


def _typhoon_request(image_path: Path, *, max_retries: int = 3, timeout: int = 60) -> str:
    """One call, with exponential backoff on transient errors."""
    api_key = os.getenv("TYPHOON_API_KEY") or TYPHOON_API_KEY
    if not api_key:
        raise RuntimeError(
            "Set TYPHOON_API_KEY before OCR is needed.\n"
            "  Bash: export TYPHOON_API_KEY='sk-...'\n"
            "  Or put it in a local .env file that is not committed."
        )

    headers = {"Authorization": f"Bearer {api_key}"}
    last_err = None

    for attempt in range(1, max_retries + 1):
        try:
            with open(image_path, "rb") as f:
                resp = requests.post(
                    TYPHOON_URL,
                    headers=headers,
                    files={"file": f},
                    data=TYPHOON_PARAMS.copy(),
                    timeout=timeout,
                )

            if resp.status_code == 200:
                payload = resp.json()
                return _extract_text_from_response(payload)

            if resp.status_code in (429, 500, 502, 503, 504):
                wait = 2 ** attempt
                log.warning(f"  Typhoon HTTP {resp.status_code}, retry {attempt}/{max_retries} in {wait}s")
                time.sleep(wait)
                last_err = f"HTTP {resp.status_code}"
                continue

            raise TyphoonError(f"HTTP {resp.status_code}: {resp.text[:200]}")

        except requests.RequestException as e:
            last_err = str(e)
            time.sleep(2 ** attempt)

    raise TyphoonError(f"Exhausted retries: {last_err}")


def ocr_page(image_path: Path) -> str:
    """OCR with disk cache. Re-running this notebook skips already-OCR\'d pages."""
    cache = OCR_CACHE_DIR / f"{image_path.stem}.txt"
    if cache.exists():
        return cache.read_text(encoding="utf-8")

    raw = _typhoon_request(image_path)
    cache.write_text(raw, encoding="utf-8")
    return raw


## 6 · Header parser — what page is this?

Every page-1 of a station\'s form has a header that names:

> รายงานผลการนับคะแนน...
> หน่วยเลือกตั้งที่ **N**, หมู่ที่ M, ตำบล/แขวง/เทศบาล **TAMBOL** อำเภอ/เขต **AMPHOE**
> เขตเลือกตั้งที่ 2, จังหวัด ชัยภูมิ

Plus a form badge (top-right):
- "ส.ส. 5/18" → constituency vote
- "ส.ส. 5/18 (บช)" → party-list vote

We extract these markers from page-1 OCR. Subsequent pages (2-3 for 5/18, 2-4 for 5/18 บช) belong to the same station and just say "- 2 -", "- 3 -" etc. with no header.


In [30]:
# Thai digit ↔ Arabic digit
THAI_NUM_MAP = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")

def thai_to_int(s) -> Optional[int]:
    """Best-effort: extract first integer from a string with Thai or Arabic digits."""
    if s is None:
        return None
    s = str(s).translate(THAI_NUM_MAP)
    m = re.search(r"\d+", s.replace(",", ""))
    return int(m.group(0)) if m else None


# ── Patterns ──────────────────────────────────────────────────────────────────

RX_FORM_PARTY = re.compile(
    r"ส[\.\s]*ส[\.\s]*\s*[5๕]\s*/\s*(?:18|๑๘)\s*\(\s*บช\s*\)",
    re.IGNORECASE
)
RX_FORM_CONSTIT = re.compile(
    r"ส[\.\s]*ส[\.\s]*\s*[5๕]\s*/\s*(?:18|๑๘)(?!\s*\()",
    re.IGNORECASE
)
RX_FORM_516 = re.compile(r"ส[\.\s]*ส[\.\s]*\s*[5๕]\s*/\s*(?:16|๑๖)", re.IGNORECASE)
RX_FORM_517 = re.compile(r"ส[\.\s]*ส[\.\s]*\s*[5๕]\s*/\s*(?:17|๑๗)", re.IGNORECASE)

# "หน่วยเลือกตั้งที่ 1" — accept Thai or Arabic digits, allow leading dots/spaces
RX_STATION_NO = re.compile(r"หน่วยเลือกตั้งที่[\s\.]*([๐-๙\d]+)")

# Introductory paragraph that ONLY appears on page 1 of each station form.
# Used as a fallback first-page marker when the station number is illegible.
RX_FIRST_PAGE_INTRO = re.compile(
    r"บัดนี้\s*คณะกรรมการประจำหน่วยเลือกตั้ง|"
    r"รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎร"
)

# "ตำบล/แขวง/เทศบาล กะฮาด" — capture word after the label
RX_TAMBOL = re.compile(r"ตำบล\s*[/]?\s*(?:แขวง\s*[/]?\s*)?(?:เทศบาล\s*)?([\u0E00-\u0E7F]+?)(?=\s+อำเภอ|\s+เขต|\s*$)")

# "อำเภอ/เขต เนินสง่า"
RX_AMPHOE = re.compile(r"อำเภอ\s*[/]?\s*(?:เขต\s+)?([\u0E00-\u0E7F]+)")

# Page numbering "- 2 -"
RX_PAGE_MARK = re.compile(r"-\s*([๐-๙\d]+)\s*-")

# PDF filename → official subdistrict name corrections.
# Some PDFs use common/local names that differ from the official name in stations.csv.
PDF_SUBDISTRICT_ALIASES = {
    "ทุ่งทอง": "ตะโกทอง",   # ต.ทุ่งทอง PDF = ตำบลตะโกทอง (อ.ซับใหญ่)
}


def detect_form_type(text: str) -> Optional[str]:
    """Identify which form this page belongs to from header text."""
    if RX_FORM_PARTY.search(text):
        return "5_18_party"
    if RX_FORM_CONSTIT.search(text):
        return "5_18"
    if RX_FORM_516.search(text):
        return "5_16_party" if "บช" in text or "บัญชีรายชื่อ" in text else "5_16"
    if RX_FORM_517.search(text):
        return "5_17_party" if "บช" in text or "บัญชีรายชื่อ" in text else "5_17"
    return None


def extract_pdf_location(source_pdf: str) -> dict:
    """
    Derive district and subdistrict from the PDF file path — far more reliable
    than OCR'd tambol/amphoe text.

    Path pattern: .../อำเภอXXX/ต.YYY-[แบ่งเขต|บัญชีรายชื่อ].pdf
    Returns {"district": ..., "subdistrict": ...}; either may be None.

    Multi-tambol files (e.g. "ทต.จัตุรัส ทต.บ้านกอก-...") return subdistrict=None
    because a single subdistrict cannot be determined from the path alone.

    PDF_SUBDISTRICT_ALIASES corrects cases where the PDF uses a common/local name
    that differs from the official name in stations.csv.
    """
    path = Path(source_pdf)
    parent = path.parent.name

    district = re.sub(r'^อำเภอ\s*', '', parent).strip() if 'อำเภอ' in parent else None
    stem = path.stem

    if stem.count('ทต.') > 1:
        return {"district": district, "subdistrict": None}

    subdistrict = None

    m = re.match(r'^ต\.([\u0E00-\u0E7F]+)', stem)
    if m:
        subdistrict = m.group(1)

    if subdistrict is None:
        m = re.match(r'^ทต\.([\u0E00-\u0E7F]+)', stem)
        if m:
            subdistrict = m.group(1)

    if subdistrict is None:
        m = re.match(r'^([\u0E00-\u0E7F]+)', stem)
        if m:
            subdistrict = m.group(1)

    if subdistrict and subdistrict in PDF_SUBDISTRICT_ALIASES:
        subdistrict = PDF_SUBDISTRICT_ALIASES[subdistrict]

    return {"district": district, "subdistrict": subdistrict}


def parse_page_header(text: str) -> dict:
    """
    Try to extract page-1 markers from OCR text.

    is_first_page is set True when EITHER:
      (a) station_no is extracted — confident first page with known station
      (b) the introductory paragraph is present but station_no is illegible —
          first page with station_no=None, flagged for review downstream
    """
    out = {
        "form_type":     detect_form_type(text),
        "station_no":    None,
        "tambol":        None,
        "amphoe":        None,
        "page_marker":   None,
        "is_first_page": False,
    }

    m = RX_STATION_NO.search(text)
    if m:
        out["station_no"] = thai_to_int(m.group(1))
        out["is_first_page"] = True
    elif RX_FIRST_PAGE_INTRO.search(text):
        out["is_first_page"] = True  # station_no unreadable but still page 1

    m = RX_TAMBOL.search(text)
    if m:
        out["tambol"] = m.group(1).strip()

    m = RX_AMPHOE.search(text)
    if m:
        out["amphoe"] = m.group(1).strip()

    m = RX_PAGE_MARK.search(text)
    if m:
        out["page_marker"] = thai_to_int(m.group(1))

    return out

## 7 · Index pages → station chunks

For each source PDF: OCR every page once, parse header, group consecutive pages into chunks per (station × form).

Stop conditions for a chunk: next page has a different form_type OR a different station_no in its header. The "signature page" (last page of the form) is included with the previous chunk because it has no station header itself.

This step is the expensive one — call Typhoon once per page. Cached on disk so re-running is free.


In [31]:
def resolve_project_path(path_like) -> Path:
    """Resolve cached relative paths from any OS to a real path under PROJECT_ROOT."""
    if path_like is None:
        return None
    path_str = str(path_like).replace("\\", "/")
    path = Path(path_str)
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path


def ocr_text_cache_exists(image_path: Path) -> bool:
    return (OCR_CACHE_DIR / f"{image_path.stem}.txt").exists()


def page_index_cache_is_usable(pages: list[dict]) -> bool:
    """Accept cached page indexes when either the rendered image or OCR text cache exists."""
    image_paths = [p.get("image_path") for p in pages if p.get("image_path")]
    if not image_paths:
        return False
    for img_rel in image_paths:
        img = resolve_project_path(img_rel)
        if not img.exists() and not ocr_text_cache_exists(img):
            return False
    return True


def index_pdf_pages(pdf_path: Path) -> list[dict]:
    """
    Walk pages of a PDF, OCR each, parse header. Returns list of page-dicts:
       {page_idx, image_path, form_type, station_no, tambol, amphoe, is_first_page, ...}
    Caches result to PAGE_INDEX_DIR/<pdf_stem>.json — re-running skips Typhoon calls.
    """
    cache = PAGE_INDEX_DIR / f"{pdf_path.stem}.json"
    if cache.exists():
        pages = json.loads(cache.read_text(encoding="utf-8"))
        if page_index_cache_is_usable(pages):
            return pages
        log.warning(f"  Ignoring stale page-index cache with missing images/OCR text: {cache.name}")

    n_pages = count_pdf_pages(pdf_path)
    pages = []
    log.info(f"Indexing {pdf_path.name} ({n_pages} pages)")

    for i in range(n_pages):
        try:
            img = render_page(pdf_path, i)
            text = ocr_page(img)
            hdr = parse_page_header(text)
            pages.append({
                "page_idx":   i,
                "image_path": img.relative_to(PROJECT_ROOT).as_posix(),
                **hdr,
                "ocr_chars":  len(text),
            })
        except Exception as e:
            log.warning(f"  Page {i} failed: {e}")
            pages.append({"page_idx": i, "image_path": None, "form_type": None,
                          "station_no": None, "tambol": None, "amphoe": None,
                          "page_marker": None, "is_first_page": False,
                          "error": str(e)})

    cache.write_text(json.dumps(pages, ensure_ascii=False, indent=2), encoding="utf-8")
    return pages


def chunk_pages_by_station(pages: list[dict]) -> list[dict]:
    """
    Group pages into per-station chunks. A new chunk starts at every is_first_page=True.
    Pages without their own header attach to the most recent chunk.
    """
    chunks = []
    current = None

    for p in pages:
        if p.get("is_first_page"):
            if current:
                chunks.append(current)
            current = {
                "form_type":   p["form_type"],
                "station_no":  p["station_no"],
                "tambol":      p["tambol"],
                "amphoe":      p["amphoe"],
                "page_indices": [p["page_idx"]],
                "image_paths":  [p["image_path"]],
            }
        elif current:
            current["page_indices"].append(p["page_idx"])
            current["image_paths"].append(p["image_path"])
        # else: orphan page before any header — skip silently

    if current:
        chunks.append(current)

    return chunks


## 8 · Body parsers — header counts + vote table

In [32]:
# Header field labels — robust to spacing variation
HEADER_FIELDS = {
    "eligible_voters":   r"จำนวนผู้มีสิทธิ.{0,20}ตามบัญชี",
    "voters_present":    r"(?:จำนวน)?ผู้มีสิทธิ.{0,20}มาแสดงตน",
    "ballots_received":  r"บัตรเลือกตั้งที่ได้รับ",
    "ballots_used":      r"บัตรเลือกตั้งที่ใช้",
    "ballots_good":      r"บัตรดี",
    "ballots_spoiled":   r"บัตรเสีย",
    "ballots_no_vote":   r"บัตรไม่เลือก",
}


def parse_chunk_header(combined_text: str) -> dict:
    """Extract header counts from the combined text of a chunk."""
    out = {}
    for field, label_rx in HEADER_FIELDS.items():
        # Match: <label> ... <digits> [บัตร|คน] ( <thai_word> )
        rx = re.compile(
            label_rx + r".{0,80}?([๐-๙\d,]+)\s*(?:บัตร|คน)?\s*\(([^)]{1,40})\)",
            re.DOTALL,
        )
        m = rx.search(combined_text)
        if m:
            out[field] = {"value": thai_to_int(m.group(1)),
                          "thai_word": m.group(2).strip()}
        else:
            out[field] = {"value": None, "thai_word": None}
    return out


def parse_vote_table(combined_text: str, *, kind: str) -> list[dict]:
    """
    Extract vote rows from all <table> blocks in OCR output.
    kind='candidate' -> entity_no maps to CANDIDATES dict
    kind='party'     -> entity_no maps to PARTIES dict
    """
    soup = BeautifulSoup(combined_text, "html.parser")
    rows = []
    seen = set()

    ref = CANDIDATES if kind == "candidate" else PARTIES

    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            if tr.find("th"):
                continue

            cells = [td.get_text(strip=True) for td in tr.find_all("td")]
            if len(cells) < 2:
                continue

            entity_no = thai_to_int(cells[0])
            if entity_no is None or entity_no not in ref:
                continue

            # Avoid duplicate OCR rows if the same party appears twice
            if entity_no in seen:
                continue
            seen.add(entity_no)

            votes_int = None
            thai_word = None

            for c in cells[1:]:
                n = thai_to_int(c)
                if n is not None and votes_int is None:
                    votes_int = n
                elif re.search(r"[\u0E00-\u0E7F]", c):
                    if not c.startswith(("พรรค", "นาย", "นาง", "นางสาว")):
                        thai_word = c.strip("()")

            rows.append({
                "entity_no": entity_no,
                "votes": votes_int,
                "votes_thai_word": thai_word,
            })

    return rows


## 9 · Validation

Two consistency checks per chunk:
1. **Digit ↔ Thai word** — does `560` match `ห้าร้อยหกสิบ`?
2. **Arithmetic sums** — does `ballots_used == ballots_good + ballots_spoiled + ballots_no_vote`? Does `sum(votes) == ballots_good`?

A failed check doesn\'t drop the row — just flags it for review (these often indicate OCR errors, occasionally indicate human filling errors on the form itself, which is itself an interesting finding).

> **Note on common OCR confusables**: As you process more pages, you\'ll observe specific Thai-word OCR errors (e.g. `น`/`ม`, `ห`/`พ`, `ย`/`ธ`). Add them to `KNOWN_OCR_FIXES` below as you find them — let them grow from your own observations.


In [33]:
# Empirically-discovered OCR fixes — start empty, grow from your own QA findings
# OCR errors we've observed in Typhoon's output for these specific forms.
# When you find new ones, add them here. The keys must match what OCR actually produces.
KNOWN_OCR_FIXES = {
    # 'X พิเศษ' / 'X ฉลาด' / 'X ปอดี' / 'X ถ้วน' — Typhoon often misreads 'ถ้วน' (final marker)
    # Usually appears after a number, ignored by word-to-int conversion anyway.
    # We strip these in normalize_thai_word() implicitly by not affecting the digit parse.
    "ปอดี": "ถ้วน",
    "พิเศษ": "ถ้วน",
    "ฉลาด": "ถ้วน",
    # 'ทำ' often misread for 'ห้า' in handwriting OCR
    "ทำร้อย": "ห้าร้อย",
}


def normalize_thai_word(w: Optional[str]) -> str:
    if not w:
        return ""
    s = w.strip().strip("()")
    for k, v in KNOWN_OCR_FIXES.items():
        s = s.replace(k, v)
    return s


def thai_word_to_int(w: Optional[str]) -> Optional[int]:
    if not w or not HAS_PYTHAINLP:
        return None
    try:
        cleaned_word = normalize_thai_word(w)
        num_result = text_to_num(cleaned_word)
        
        # FIX: If text_to_num returns a list, extract the first item
        if isinstance(num_result, list):
            if not num_result: # Handle empty lists
                return None
            num_result = num_result[0]
            
        return int(num_result)
        
    # ADDED TypeError to prevent the script from completely crashing in the future
    except (ValueError, KeyError, IndexError, TypeError):
        return None


def cross_check(value: Optional[int], thai_word: Optional[str]) -> tuple[bool, str]:
    """Return (ok, reason). Both missing is OK (=both_missing, treated as info not error)."""
    if value is None and not thai_word:
        return True, "both_missing"
    if value is None:
        return False, "digit_missing"
    if not thai_word:
        return False, "word_missing"
    converted = thai_word_to_int(thai_word)
    if converted is None:
        return False, f"unparseable_word:{thai_word}"
    if converted != value:
        return False, f"mismatch:{value}!={converted}({thai_word})"
    return True, ""


def validate_chunk(header: dict, votes: list[dict]) -> tuple[list[str], str]:
    """Run all checks. Returns (flag_list, status)."""
    flags = []

    # Digit↔word for header fields
    for field, rec in header.items():
        ok, reason = cross_check(rec["value"], rec["thai_word"])
        if not ok:
            flags.append(f"hdr_{field}:{reason}")

    H = {k: v["value"] for k, v in header.items()}

    # ballots_used == good + spoiled + no_vote
    if all(H.get(k) is not None for k in ("ballots_used", "ballots_good", "ballots_spoiled", "ballots_no_vote")):
        s = H["ballots_good"] + H["ballots_spoiled"] + H["ballots_no_vote"]
        if s != H["ballots_used"]:
            flags.append(f"sum_used!=g+s+nv ({H['ballots_used']}!={s})")

    # sum(votes) == ballots_good
    if H.get("ballots_good") is not None:
        total = sum((v.get("votes") or 0) for v in votes)
        if total != H["ballots_good"]:
            flags.append(f"sum_votes!=good ({total}!={H['ballots_good']})")

    # Per-vote digit↔word
    for v in votes:
        ok, _ = cross_check(v.get("votes"), v.get("votes_thai_word"))
        v["needs_review"] = not ok

    status = "ok" if not flags else "needs_review"
    return flags, status


## 10 · Process one chunk → station record + vote rows

In [34]:
def lookup_station_metadata(form_type: str, station_no, tambol: Optional[str],
                             source_pdf: Optional[str] = None) -> dict:
    """Match (station_no, tambol) → station_code from stations.csv.

    Resolution order (most-to-least reliable):
      1. station_no + subdistrict from source PDF path  ← most reliable
      2. station_no + OCR tambol (often garbled by Typhoon)
      3. station_no + district from PDF path (multi-tambol files land here)
      4. First candidate by station_no alone (last resort — marked with '?')

    Root cause of the old "always บ้านเขว้า" bug:
      The old fallback used `iloc[0]` on the full dataframe.  Since stations.csv
      is sorted with อำเภอบ้านเขว้า first, every failed tambol-match returned a
      บ้านเขว้า row.  The fix is to anchor on the PDF path instead.
    """
    if form_type not in ("5_18", "5_18_party") or station_no is None:
        return {"station_code": None, "district": None, "subdistrict": None}

    # Reliable location extracted from the PDF file path
    pdf_loc = extract_pdf_location(source_pdf) if source_pdf else {"district": None, "subdistrict": None}

    candidates = stations_df[stations_df["station_no"].astype(int) == int(station_no)]
    if candidates.empty:
        return {"station_code": None, "district": None, "subdistrict": None}

    def _pick(df, uncertain=False):
        r = df.iloc[0]
        mark = "?" if (uncertain or len(df) > 1) else ""
        return {"station_code": str(r["station_code"]) + mark,
                "district":     r["district"],
                "subdistrict":  r["subdistrict"]}

    # ── 1. station_no + PDF subdistrict ──────────────────────────────────────
    if pdf_loc.get("subdistrict"):
        m = candidates[candidates["subdistrict"] == pdf_loc["subdistrict"]]
        if not m.empty:
            return _pick(m)

    # ── 2. station_no + OCR tambol ────────────────────────────────────────────
    if tambol:
        m = candidates[candidates["subdistrict"] == tambol]
        if len(m) == 1:
            return _pick(m)

    # ── 3. station_no + PDF district (multi-tambol files) ────────────────────
    if pdf_loc.get("district"):
        m = candidates[candidates["district"] == pdf_loc["district"]]
        if not m.empty:
            # OCR tambol as tiebreaker when multiple candidates in the same district
            if tambol and len(m) > 1:
                m2 = m[m["subdistrict"] == tambol]
                if len(m2) == 1:
                    return _pick(m2)
            return _pick(m)

    # ── 4. Last resort ────────────────────────────────────────────────────────
    return _pick(candidates, uncertain=True)


def process_chunk(chunk: dict, source_pdf: str) -> dict:
    """
    Take an indexed chunk, fetch all its OCR'd text, parse + validate.
    Returns dict: {station: {...}, votes: [...], error: ...}
    """
    form_type = chunk["form_type"]
    if form_type is None:
        return {"error": "form_type_unknown", "station": None, "votes": []}

    # Combine OCR text from all pages of this chunk
    page_texts = []
    missing_images = []
    for img_rel in chunk["image_paths"]:
        if img_rel is None:
            continue
        img = resolve_project_path(img_rel)
        if not img.exists() and not ocr_text_cache_exists(img):
            missing_images.append(str(img))
            continue
        try:
            page_texts.append(ocr_page(img))
        except Exception as e:
            log.warning(f"  OCR retrieval failed for {img}: {e}")
    combined = "\n".join(page_texts)

    if not combined.strip():
        sample = missing_images[:3]
        return {
            "error": "ocr_text_missing",
            "station": None,
            "votes": [],
            "missing_images": sample,
        }

    # Resolve station identity — pass source_pdf so lookup can use path-derived location
    meta = lookup_station_metadata(form_type, chunk["station_no"], chunk["tambol"], source_pdf)

    # Parse body
    header = parse_chunk_header(combined)
    kind = "party" if form_type.endswith("_party") else "candidate"
    votes_raw = parse_vote_table(combined, kind=kind)
    flags, status = validate_chunk(header, votes_raw)

    # If station_no couldn't be read from the form, flag it explicitly
    if chunk["station_no"] is None:
        flags = ["station_no_unreadable"] + flags
        status = "needs_review"

    station_record = {
        "province":         PROVINCE,
        "constituency_no":  CONSTITUENCY_NO,
        "form_type":        form_type,
        "station_code":     meta["station_code"],
        "station_no":       chunk["station_no"],
        "district":         meta["district"],
        "subdistrict":      meta["subdistrict"] or chunk["tambol"],
        **{k: v["value"] for k, v in header.items()},
        "source_pdf":       source_pdf,
        "source_pages":     ",".join(str(p) for p in chunk["page_indices"]),
        "ocr_status":       status,
        "validation_flags": ";".join(flags),
    }

    vote_rows = []
    ref = CANDIDATES if kind == "candidate" else PARTIES
    for v in votes_raw:
        no = v["entity_no"]
        if kind == "candidate":
            name, party = ref.get(no, ("?", "?"))
        else:
            name, party = ref.get(no, "?"), None
        vote_rows.append({
            "province":         PROVINCE,
            "constituency_no":  CONSTITUENCY_NO,
            "form_type":        form_type,
            "station_code":     meta["station_code"],
            "station_no":       chunk["station_no"],
            "district":         meta["district"],
            "subdistrict":      meta["subdistrict"] or chunk["tambol"],
            "entity_kind":      kind,
            "entity_no":        no,
            "entity_name":      name,
            "party_name":       party,
            "votes":            v.get("votes"),
            "votes_thai_word":  v.get("votes_thai_word"),
            "needs_review":     v.get("needs_review", False),
        })

    return {"station": station_record, "votes": vote_rows, "error": None}


## 11 · Batch run — process every source PDF

Iterates over unique source PDFs from the manifest. For each: index pages, chunk by station, process every chunk.

Resumable — checkpoint per (PDF + chunk) saved as JSON. Re-running picks up where it left off.

For ~30 PDFs × ~50-200 pages each → expect a few hundred Typhoon calls. Cached on disk so repeated runs skip everything that\'s already done.


In [35]:
def chunk_checkpoint_path(source_pdf_rel: str, station_no, form_type: str,
                           first_page_idx: int = 0) -> Path:
    safe = source_pdf_rel.replace("/", "__").replace("\\", "__").replace(".pdf", "").replace(".PDF", "")
    # Use page index as surrogate key when station_no is unreadable
    st_label = str(station_no) if station_no is not None else f"unkn{first_page_idx}"
    return STATION_OUT / f"{safe}__{form_type}__st{st_label}.json"


def run_batch(*, force: bool = False, limit_pdfs: Optional[int] = None,
              filter_kind: Optional[str] = None):
    """
    Process every unique source PDF in the manifest.
       force=True       — re-process even if checkpoint exists
       limit_pdfs=N     — only do first N PDFs (smoke test)
       filter_kind=...  — only PDFs of this source_kind ('pure', 'mixed', 'constituency')

    Chunks with station_no=None (illegible station number) are now saved with a
    placeholder key (unkn<page_idx>) and flagged 'station_no_unreadable' so they
    appear in the QA report rather than being silently dropped.
    """
    # Unique source PDFs (drop missing)
    sources = manifest_df.dropna(subset=["source_pdf"]).copy()
    if filter_kind:
        sources = sources[sources["source_kind"] == filter_kind]
    unique_pdfs = sources["source_pdf"].unique()
    if limit_pdfs:
        unique_pdfs = unique_pdfs[:limit_pdfs]

    n_pdfs = len(unique_pdfs)
    n_chunks_done = 0
    n_chunks_failed = 0

    for i, src_rel in enumerate(unique_pdfs, 1):
        pdf = PROJECT_ROOT / src_rel
        log.info(f"[{i}/{n_pdfs}] {src_rel}")

        if not pdf.exists():
            log.warning(f"  Missing: {pdf}")
            continue

        try:
            pages = index_pdf_pages(pdf)
            chunks = chunk_pages_by_station(pages)
            log.info(f"  Found {len(chunks)} chunks across {len(pages)} pages")
        except Exception as e:
            log.exception(f"  Indexing failed: {e}")
            continue

        for chunk in chunks:
            # Skip chunks with no form type (can't determine what form this is)
            if not chunk.get("form_type"):
                continue
            # station_no=None is now allowed — saved with unknN key and flagged for review
            st = chunk.get("station_no")
            first_page = chunk["page_indices"][0] if chunk.get("page_indices") else 0
            cp = chunk_checkpoint_path(src_rel, st, chunk["form_type"], first_page)
            if cp.exists() and not force:
                continue
            try:
                result = process_chunk(chunk, src_rel)
                cp.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
                if result.get("error"):
                    n_chunks_failed += 1
                else:
                    n_chunks_done += 1
            except Exception as e:
                log.exception(f"  Chunk failed: station={st} form={chunk.get('form_type')}")
                n_chunks_failed += 1

    log.info(f"Batch done. Processed {n_chunks_done} chunks, {n_chunks_failed} failed.")


In [36]:
# Guard heavy OCR runs when using Run All.
# Default behavior is a 1-PDF smoke test without force; set RUN_FULL_OCR=True only when ready.
RUN_FULL_OCR = False
RUN_OCR_SMOKE_TEST = True
OCR_FORCE = False
SMOKE_LIMIT_PDFS = 1

if RUN_FULL_OCR:
    run_batch(force=OCR_FORCE)
elif RUN_OCR_SMOKE_TEST:
    run_batch(force=OCR_FORCE, limit_pdfs=SMOKE_LIMIT_PDFS)
else:
    print('Skipping OCR batch. Set RUN_OCR_SMOKE_TEST=True or RUN_FULL_OCR=True to run it.')

2026-05-08 20:48:38,178 INFO | [1/34] data/raw/pdfs_gdrive/อำเภอบ้านเขว้า/ต.บ้านเขว้า-แบ่งเขต-บัญชีรายชื่อ.pdf
2026-05-08 20:48:38,181 INFO |   Found 46 chunks across 140 pages
2026-05-08 20:48:38,562 INFO | [2/34] data/raw/pdfs_gdrive/อำเภอบ้านเขว้า/ต.ตลาดแร้ง-แบ่งเขต_บัญชีรายชื่อ.pdf
2026-05-08 20:48:38,564 INFO |   Found 46 chunks across 142 pages
2026-05-08 20:48:38,725 INFO | [3/34] data/raw/pdfs_gdrive/อำเภอบ้านเขว้า/ต.ลุ่มลำชี แบ่งเขต.PDF
2026-05-08 20:48:38,726 INFO |   Found 23 chunks across 46 pages
2026-05-08 20:48:38,769 INFO | [4/34] data/raw/pdfs_gdrive/อำเภอบ้านเขว้า/ต.ลุ่มน้ำชี-บัญชีรายชื่อ.pdf
2026-05-08 20:48:38,771 INFO |   Found 23 chunks across 92 pages
2026-05-08 20:48:38,902 INFO | [5/34] data/raw/pdfs_gdrive/อำเภอบ้านเขว้า/ต.ชีบน-แบ่งเขต_บัญชีรายชื่อ.pdf
2026-05-08 20:48:38,903 INFO |   Found 26 chunks across 77 pages
2026-05-08 20:48:38,994 INFO | [6/34] data/raw/pdfs_gdrive/อำเภอบ้านเขว้า/ต.ภูแลนคา-แบ่งเขต.PDF
2026-05-08 20:48:38,995 INFO |   Found 10 chunks a

## 12 · Aggregate checkpoints → final tables

In [37]:
def aggregate_checkpoints():
    station_rows, vote_rows = [], []
    n_files = 0
    for cp in STATION_OUT.glob("*.json"):
        try:
            data = json.loads(cp.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            log.warning(f"Bad JSON: {cp.name}")
            continue
        n_files += 1
        if data.get("station"):
            station_rows.append(data["station"])
        vote_rows.extend(data.get("votes", []))

    log.info(f"Aggregated {n_files} checkpoint files → {len(station_rows)} station records, {len(vote_rows)} vote rows")

    stations = pd.DataFrame(station_rows)
    votes    = pd.DataFrame(vote_rows)

    # Type cleanup
    for c in ["constituency_no", "station_no", "eligible_voters", "voters_present",
              "ballots_received", "ballots_used", "ballots_good",
              "ballots_spoiled", "ballots_no_vote"]:
        if c in stations.columns:
            stations[c] = pd.to_numeric(stations[c], errors="coerce").astype("Int64")

    if "votes" in votes.columns:
        votes["votes"]     = pd.to_numeric(votes["votes"],     errors="coerce").astype("Int64")
        votes["entity_no"] = pd.to_numeric(votes["entity_no"], errors="coerce").astype("Int64")

    # Save
    stations.to_parquet(PROCESSED_DIR / "stations.parquet", index=False)
    stations.to_csv(PROCESSED_DIR / "stations.csv", index=False, encoding="utf-8-sig")
    votes.to_parquet(PROCESSED_DIR / "votes.parquet", index=False)
    votes.to_csv(PROCESSED_DIR / "votes.csv", index=False, encoding="utf-8-sig")

    print(f"Wrote stations.parquet ({len(stations)} rows), votes.parquet ({len(votes)} rows)")
    return stations, votes


stations_df_out, votes_df_out = aggregate_checkpoints()


2026-05-08 20:48:40,983 INFO | Aggregated 619 checkpoint files → 619 station records, 19651 vote rows


Wrote stations.parquet (619 rows), votes.parquet (19651 rows)


## 13 · QA report

In [38]:
def qa_report():
    if not (PROCESSED_DIR / "stations.parquet").exists():
        print("Run aggregate_checkpoints() first.")
        return

    s = pd.read_parquet(PROCESSED_DIR / "stations.parquet")
    v = pd.read_parquet(PROCESSED_DIR / "votes.parquet")

    print("OCR status:")
    print(s["ocr_status"].value_counts().to_string())
    print()

    print("By form type × status:")
    print(s.groupby(["form_type", "ocr_status"]).size().unstack(fill_value=0).to_string())
    print()

    flags = s["validation_flags"].dropna().astype(str).str.split(r"[;|]", regex=True).explode().str.strip()
    flags = flags[flags.str.len() > 0]
    if len(flags):
        print(f"Top 15 validation flags ({len(flags)} total):")
        # Just take the prefix before the first ":"
        flag_types = flags.str.split(":").str[0]
        print(flag_types.value_counts().head(15).to_string())
        print()

    # Compare totals against ground truth
    ref = json.loads((EXTERNAL_DIR / "reference_constituency.json").read_text(encoding="utf-8"))
    ref_total_constit = ref["constituency"]["total_candidate_votes"]
    ref_total_party   = ref["party_list"]["total_party_votes"]

    our_total_constit = v[v["form_type"] == "5_18"]["votes"].sum()
    our_total_party   = v[v["form_type"] == "5_18_party"]["votes"].sum()

    print("=" * 60)
    print("GROUND TRUTH COMPARISON")
    print("=" * 60)
    print(f"Constituency votes:")
    print(f"  Reference (PBS):  {ref_total_constit:>10,}")
    print(f"  Our OCR:          {our_total_constit:>10,}")
    print(f"  Coverage:         {our_total_constit/ref_total_constit*100:.1f}%")
    print()
    print(f"Party-list votes:")
    print(f"  Reference (PBS):  {ref_total_party:>10,}")
    print(f"  Our OCR:          {our_total_party:>10,}")
    print(f"  Coverage:         {our_total_party/ref_total_party*100:.1f}%")
    print()

    # Save needs-review subset for manual fixing
    needs_review = s[s["ocr_status"] != "ok"]
    needs_review.to_csv(PROCESSED_DIR / "qa_needs_review.csv", index=False, encoding="utf-8-sig")
    print(f"Stations needing review: {len(needs_review)} -> qa_needs_review.csv")


qa_report()


OCR status:
ocr_status
needs_review    554
ok               65

By form type × status:
ocr_status  needs_review  ok
form_type                   
5_18                 269  40
5_18_party           285  25

Top 15 validation flags (1788 total):
validation_flags
hdr_eligible_voters           322
hdr_voters_present            241
hdr_ballots_used              241
hdr_ballots_received          236
hdr_ballots_good              213
hdr_ballots_spoiled           178
hdr_ballots_no_vote             6
sum_votes!=good (238!=239)      2
sum_votes!=good (148!=147)      2
sum_votes!=good (197!=8)        2
sum_votes!=good (436!=181)      2
sum_votes!=good (219!=199)      2
sum_votes!=good (60!=70)        2
sum_votes!=good (123!=127)      2
sum_votes!=good (163!=221)      2

GROUND TRUTH COMPARISON
Constituency votes:
  Reference (PBS):      79,326
  Our OCR:              71,539
  Coverage:         90.2%

Party-list votes:
  Reference (PBS):      79,374
  Our OCR:              96,495
  Coverage:      

---

## ✅ When this notebook finishes

| Output | Description |
|---|---|
| `data/processed/ocr_cache/*.txt` | Raw OCR text per page (cached) |
| `data/processed/page_index/*.json` | Indexed pages per source PDF (cached) |
| `data/processed/stations_raw/*.json` | Per-chunk results (resumable) |
| `data/processed/stations.parquet` + `.csv` | Final station table |
| `data/processed/votes.parquet` + `.csv` | Final vote table (long format) |
| `data/processed/qa_needs_review.csv` | Rows flagged by validation |

## Recommended workflow

1. Set `TYPHOON_API_KEY` in environment
2. **Smoke test 1 PDF** — `run_batch(limit_pdfs=1)`. Inspect cached files, eyeball a JSON output. If header/votes look right, parsers are working.
3. **Tune** — common things you may need to adjust:
    - regex in `parse_chunk_header` if Typhoon formats labels differently than expected
    - `parse_vote_table` heuristics if the table column order differs
    - Add common OCR errors to `KNOWN_OCR_FIXES`
4. **Full run** — `run_batch()` (no limit). Will take ~30-60 minutes for ~30 PDFs.
5. **Aggregate** — `aggregate_checkpoints()`
6. **QA** — `qa_report()` and act on the flagged rows
7. Hand off to nb03 (cleaning) and nb04 (analysis)
